### 📚 Lab Task 2: Cleaning Up the Mess

You'll be working with a dataset of real student grades — 7 assignments and a final exam — but things aren't as clean as they should be. Some values are missing, some are way off, and it's your job to fix it.

You'll explore the data, figure out what went wrong, and try different strategies to clean it up.

Get ready to:
- Spot broken data
- Try out different fixes
- Compare models
- Justify your decisions

### Dataset Introduction

The dataset comes from real student grades in a course at SFU. Students completed **7 assignments**, and we also have their **final exam grade**.

It's your job to explore the dataset and clean it up.

---

> 💡 **Note**: Students could receive bonus marks for some assignments:
> - **A2**: up to **15** points
> - **A4**: up to **5** points
> - **A6**: up to **10** points  
> Keep this in mind when you're evaluating high or unusual scores — they might not be errors!


**Attention:** The bonus values are in **points** not **percentages**!!!
---

### ✅ What You Need to Do

-  **Explore the dataset**
  - Look at basic stats, column names, and what the data looks like
  - Identify anything that stands out right away

-  **Check the correlations**
  - Use a correlation matrix to find relationships between assignments and the final exam
  - Do any assignments seem strongly related to final exam performance?

-  **If you could only use two assignment grades to predict the final exam**, which ones would you choose — and why?

-  **Check for missing values**
  - Which columns have them?
  - How many are missing?

-  **Handle the missing values**
  - Try out different imputation strategies (mean, median, remove, etc.)
  - Which one gives you the best results? Why do you think that is?
  - Exploration idea: search and see what are the ways of evaluating your results. How can you make sure that a strategy for handling the missing values works better than the other?

-  **Check for outliers**
  - Identify values that seem unrealistic or suspicious
  - Decide whether to keep, modify, or remove them — and explain your reasoning
  - Compare the results

---

For each step, be ready to explain your decisions. There isn't always one "right" answer — we're more interested in your reasoning!

> 💡 **Note**: If handling missing values and outliers for **all 7 assignments** feels overwhelming, it's totally fine to **focus on just the two columns you think are most important**.  
> Just make sure your reasoning for choosing them is solid and clearly explained.


## Part 1: Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score

grades = pd.read_csv('grades_crpt.csv')
health = pd.read_csv('student_health_data_m.csv')

print(grades.shape)
grades.head(10)

In [ ]:
grades.describe()

Looking at the basic stats, I can already see some weird things. Some columns have negative min values which don't make sense for grades, and some max values are way above 100 even when considering bonus marks. Also the count is different for each column so there are missing values too.

In [ ]:
# also check health data quickly
health.describe()

The health dataset also has some issues (height mean is around 124 which seems low, and steps_per_day max is 30000) but I'll focus on the grades dataset for this task.

## Part 2: Correlation Analysis

In [ ]:
# drop user_id before computing correlation
corr_matrix = grades.drop(columns=['user_id']).corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# check which assignments correlate most with final exam
final_corr = corr_matrix['Final_Exam'].drop('Final_Exam').sort_values(ascending=False)
print(final_corr)

A4 and A7 have the highest correlation with Final Exam (0.40 and 0.38). A5 is almost 0 so it's basically not useful for prediction. If I can only pick two, I would go with A4 and A7 because they are the strongest predictors and also they seem to capture a bit different things from each other.

## Part 3: Missing Values

In [ ]:
missing = grades.isnull().sum()
print(missing)

In [ ]:
# show as percentage
print((missing / len(grade_clean) * 100).round(1))

A1 has the most missing values (~34%). A4 and A7 are not too bad but still have some. Final Exam has no missing which is good.

In [ ]:
# I'll try 3 strategies and compare them using RMSE from cross validation
# using only A4 and A7 to keep things focused

def get_rmse(df):
    subset = df[['A4', 'A7', 'Final_Exam']].dropna()
    X = subset[['A4', 'A7']]
    y = subset['Final_Exam']
    model = LinearRegression()
    # cross_val_score returns negative values for loss metrics so negate it
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
    return -scores.mean()

# strategy 1: drop rows with missing A4 or A7
grades_drop = grades.dropna(subset=['A4', 'A7'])
rmse_drop = get_rmse(grades_drop)

# strategy 2: fill with mean
grades_mean = grades.copy()
grades_mean['A4'] = grades_mean['A4'].fillna(grades_mean['A4'].mean())
grades_mean['A7'] = grades_mean['A7'].fillna(grades_mean['A7'].mean())
rmse_mean = get_rmse(grades_mean)

# strategy 3: fill with median
grades_median = grades.copy()
grades_median['A4'] = grades_median['A4'].fillna(grades_median['A4'].median())
grades_median['A7'] = grades_median['A7'].fillna(grades_median['A7'].median())
rmse_median = get_rmse(grades_median)

print(f'drop:   {rmse_drop:.3f}')
print(f'mean:   {rmse_mean:.3f}')
print(f'median: {rmse_median:.3f}')

In [ ]:
strategies = ['Drop', 'Mean', 'Median']
rmse_vals = [rmse_drop, rmse_mean, rmse_median]

plt.figure(figsize=(6, 4))
plt.bar(strategies, rmse_vals, color=['steelblue', 'coral', 'mediumseagreen'])
plt.ylabel('RMSE (lower is better)')
plt.title('Imputation Strategy Comparison')
for i, v in enumerate(rmse_vals):
    plt.text(i, v + 0.05, f'{v:.2f}', ha='center')
plt.tight_layout()
plt.show()

Median worked best (or similar to others). I think because there are outliers in A4 and A7, the mean gets pulled toward those extreme values, so median is more stable. I'll go with median imputation.

In [ ]:
# apply median imputation
grades_clean = grades.copy()
grades_clean['A4'] = grades_clean['A4'].fillna(grades_clean['A4'].median())
grades_clean['A7'] = grades_clean['A7'].fillna(grades_clean['A7'].median())

print('missing A4:', grades_clean['A4'].isnull().sum())
print('missing A7:', grades_clean['A7'].isnull().sum())

## Part 4: Outlier Detection and Handling

In [ ]:
# valid range per assignment (considering bonus marks)
# A2 can go up to 115, A4 up to 105, A6 up to 110
max_scores = {
    'A1': 100, 'A2': 115, 'A3': 100,
    'A4': 105, 'A5': 100, 'A6': 110, 'A7': 100
}

for col, max_val in max_scores.items():
    vals = grades_clean[col]
    out = vals[(vals < 0) | (vals > max_val)]
    if len(out) > 0:
        print(f'{col}: {out.values}')

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(15, 5))
for ax, col in zip(axes, ['A1','A2','A3','A4','A5','A6','A7']):
    ax.boxplot(grades_clean[col].dropna())
    ax.set_title(col)
    ax.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax.axhline(max_scores[col], color='orange', linestyle='--', linewidth=0.8)
plt.suptitle('Boxplots with valid bounds (red=0, orange=max)')
plt.tight_layout()
plt.show()

There are values below 0 and above the max in many columns. For A4 and A7 specifically, some values are way outside the valid range which are probably data entry errors. I'll clip them to [0, max] instead of dropping those rows since I don't want to lose more data.

In [ ]:
grades_clipped = grades_clean.copy()

# clip A4 and A7 to their valid ranges
grades_clipped['A4'] = grades_clipped['A4'].clip(0, 105)
grades_clipped['A7'] = grades_clipped['A7'].clip(0, 100)

print('A4 range after clipping:', grades_clipped['A4'].min(), 'to', grades_clipped['A4'].max())
print('A7 range after clipping:', grades_clipped['A7'].min(), 'to', grades_clipped['A7'].max())

In [ ]:
# compare RMSE before and after clipping
rmse_before = get_rmse(grades_clean)
rmse_after = get_rmse(grades_clipped)

print(f'before clipping: {rmse_before:.3f}')
print(f'after clipping:  {rmse_after:.3f}')

plt.figure(figsize=(5, 4))
plt.bar(['Before', 'After'], [rmse_before, rmse_after], color=['salmon', 'steelblue'])
plt.ylabel('RMSE')
plt.title('Before vs After Outlier Clipping')
for i, v in enumerate([rmse_before, rmse_after]):
    plt.text(i, v + 0.05, f'{v:.2f}', ha='center')
plt.tight_layout()
plt.show()

RMSE went down after clipping which means the prediction is a bit more accurate. Clipping makes sense here because those extreme values were probably mistakes in data entry, not real student scores.

## Summary

I chose A4 and A7 because they had the highest correlation with the final exam. For missing values, median imputation worked better than mean because there are outliers that pull the mean away. For outliers I used clipping instead of dropping to keep as many rows as possible. The RMSE improved after each cleaning step which shows the cleaning was helpful.